# Exercise: Extend and Evaluate a LangChain RAG Pipeline

In the lesson you built a RAG pipeline with LangChain. You start from the working pipeline and take it further: make its answers **traceable** by returning their sources, **tune** how much context it retrieves, and then **evaluate** the result so you can say which setup is better, not just guess.

For the concepts behind evaluation, see [RAG Concepts](1_rag_concepts.md).

> **Our run (2026-11-27): a local model.** This notebook was run with a free model on this
> computer instead of the Groq cloud model the text names: `qwen3.5:4b`, served by
> [Ollama](https://ollama.com) (4.2 billion parameters, a 3.3 GB file). The embedding model
> `all-mpnet-base-v2` is local too. No API key was used and no text left the computer.
> Where the lesson says "Groq" or "gpt-oss", read "the local model". A small model is
> less precise than a large cloud model, so check every answer against the retrieved passages.

In [1]:
# Our run: pick the local Ollama model for this notebook process only.
# The .env file is not touched: variables set here win over it.
import os

os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen3.5:4b"

## Learning Objectives

At the end of this notebook, you should be able to:

- Extend a RAG pipeline to return the source passages behind each answer.
- Tune retrieval (the number of chunks) and observe its effect on the answer.
- Evaluate and compare two configurations with an LLM-as-judge (faithfulness and answer relevance).

## Provided: the pipeline from the lesson

Run this cell as is. It gives you `build_chain(pdf_path, k=..., chunk_size=..., chunk_overlap=...)`, which returns a ready retrieval chain, and `ask(question, chain)`, which invokes a chain and returns the full result (the `input`, the retrieved `context`, and the `answer`). It also builds `react_chain` over the React paper for you. You do not need to change anything here.

In [2]:
from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from pypdf import PdfReader
from qdrant_client.models import Distance

from llm_config import create_chat_model, validate_response

In [3]:
load_dotenv()

True

In [4]:
llm = create_chat_model(temperature=0)

In [5]:
def build_chain(pdf_path, k=4, chunk_size=200, chunk_overlap=50):
    """Load a PDF, build a Qdrant retriever, and wire it to the LLM. Returns a retrieval chain."""
    reader = PdfReader(pdf_path)

    docs = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": pdf_path, "page": i},
        )
        for i, page in enumerate(reader.pages, start=1)
    ]

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(docs)

    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )

    store = QdrantVectorStore.from_documents(
        chunks,
        embedding,
        location=":memory:",
        collection_name="paper",
        distance=Distance.COSINE,
    )

    retriever = store.as_retriever(search_kwargs={"k": k})

    prompt = Client().pull_prompt(
        "langchain-ai/retrieval-qa-chat",
        include_model=False,
        dangerously_pull_public_prompt=True,
    )

    combine = create_stuff_documents_chain(llm=llm | validate_response, prompt=prompt)
    return create_retrieval_chain(retriever=retriever, combine_docs_chain=combine)


def ask(question, chain):
    """Invoke a chain and return the full result dict (input, context, answer)."""
    return chain.invoke({"input": question})


# Build the pipeline over the React paper, ready to use
react_chain = build_chain("documents/react_paper.pdf")

## Exercise 1: Make answers traceable with source citations

A RAG answer is only trustworthy if you can check where it came from. The result from `ask` includes a `context` list of the retrieved documents, and each document carries metadata, including its page number in `doc.metadata`.

Write a function that asks a question and returns the answer **together with the source pages** of the chunks that supported it.

Hints:

- Call `ask(question, react_chain)` and read `result["answer"]` and `result["context"]`.
- Each item in `context` is a `Document`; look at `doc.metadata` to find the page.

In [6]:
def answer_with_sources(question):
    """Return the answer plus the source pages of the retrieved chunks."""
    result = ask(question, react_chain)
    # every retrieved chunk is a Document; its page number is in the metadata
    chunks = [(doc.metadata["page"], doc.page_content) for doc in result["context"]]
    return {
        "answer": result["answer"],
        # each page once, in reading order
        "pages": sorted({page for page, _ in chunks}),
        # the passages themselves, so a reader can check the answer
        "chunks": chunks,
    }

In [7]:
QUESTION = "What accuracy does ReAct reach on Fever compared with CoT?"

sourced = answer_with_sources(QUESTION)
print("Q:", QUESTION)
print("A:", sourced["answer"])
print("source pages:", sourced["pages"])
for page, text in sourced["chunks"]:
    print(f"\n--- page {page}\n{text}")

Q: What accuracy does ReAct reach on Fever compared with CoT?
A: Based on the provided context, **ReAct reaches an accuracy of 60.9** on Fever, while **CoT reaches 56.3**. Therefore, ReAct outperforms CoT by 4.6 points on this dataset.
source pages: [5, 6]

--- page 6
ReAct vs. CoT On the other hand, ReAct outperforms CoT on Fever (60.9 vs. 56.3) and slightly
lags behind CoT on HotpotQA (27.4 vs. 29.4). Fever claims for SUPPORTS/REFUTES might only

--- page 5
input questions/claims. More details are in Appendix B.1.
3.3 R ESULTS AND OBSERVATIONS
ReAct outperforms Act consistently Table 1 shows HotpotQA and Fever results using PaLM-

--- page 5
Act 25.7 58.9
ReAct 27.4 60.9
CoT-SC→ ReAct 34.2 64.6
ReAct→ CoT-SC 35.1 62.0
Supervised SoTAb 67.5 89.5
Table 1: PaLM-540B prompting results on
HotpotQA and Fever.

--- page 5
HotpotQA and Fever.
aHotpotQA EM is 27.1, 28.9, 33.8 for Standard, CoT,
CoT-SC in Wang et al. (2022b).
b(Zhu et al., 2021; Lewis et al., 2020)
0 5 10 15 20
#CoT-SC trials


With the page numbers, the answer can be checked in the PDF. This cell reads page 6 of the paper and prints the sentence with the two numbers.

In [8]:
page_6 = PdfReader("documents/react_paper.pdf").pages[5].extract_text()
start = page_6.find("ReAct outperforms CoT on Fever")
end = page_6.find(")", start) + 1
print(page_6[start:end])

ReAct outperforms CoT on Fever (60.9 vs. 56.3)


### Extra (my own): the same question with the book closed

What does the model say **without** retrieval? This cell sends only the question, with no context. Nothing here comes from the paper: the model answers from what it memorised in training.

In [9]:
closed_book = llm.invoke(QUESTION).text
print(f"{len(closed_book):,} characters. The first 900:\n")
print(closed_book[:900])

2,144 characters. The first 900:

Based on the original research paper **"ReAct: Synergizing Reasoning and Acting with Language Models"** (Yao et al., 2023), here is the performance comparison between **ReAct** and **Chain-of-Thought (CoT)** on the **Fever** dataset.

The study evaluates both methods on three sub-tasks of Fever:
1.  **Fact Verification**: Determining if a claim is true, false, or unknown based on evidence.
2.  **Claim Generation**: Generating a claim from a given context.
3.  **Entailment**: Determining if a hypothesis follows from the premise.

### Performance Comparison

In the experiments conducted using GPT-3.5 (the model specified in the paper), **ReAct significantly outperformed CoT** on the Fever dataset, particularly in tasks requiring multi-step reasoning and tool usage.

*   **Fact Verification**: ReAct achieved an accuracy of approximately **86.7%**, whereas CoT reached roughly **79.5%**.
*   


**Open book against closed book.**

- **With retrieval** the model answers 60.9 for ReAct and 56.3 for CoT, and the function returns the source pages 5 and 6. The check cell finds exactly that sentence on page 6 of the paper.
- **Without retrieval** the same model writes 2,144 characters that sound expert and are made up: an accuracy of "approximately 86.7%" against "79.5%", three sub-tasks of Fever, an experiment with GPT-3.5. None of this is in the paper.

That is the difference retrieval makes. It is also why the source pages matter: they are what lets a reader catch an invented answer.

## Exercise 2: Tune how much context you retrieve

The retriever returns the top `k` chunks. Too few may miss the answer, too many may dilute the prompt with irrelevant text. `build_chain` accepts a `k` argument, so you can build chains with different values.

Build two chains over the React paper, one with a small `k` and one with a larger `k`, ask both the same question, and compare the answers.

Hint: `chain_small = build_chain("documents/react_paper.pdf", k=2)`.

In [10]:
chain_small = build_chain("documents/react_paper.pdf", k=2)
chain_large = build_chain("documents/react_paper.pdf", k=6)

K_QUESTION = (
    "By how much does ReAct outperform imitation and reinforcement "
    "learning methods on ALFWorld and WebShop?"
)
result_small = ask(K_QUESTION, chain_small)
result_large = ask(K_QUESTION, chain_large)

for name, result in [("k=2", result_small), ("k=6", result_large)]:
    pages = [doc.metadata["page"] for doc in result["context"]]
    print(f"===== {name}: {len(pages)} chunks, from pages {pages}")
    print(result["answer"], "\n")

===== k=2: 2 chunks, from pages [1, 8]
Based on the provided context, it is not possible to determine **by how much** ReAct outperforms imitation and reinforcement learning methods.

The text states that "ReAct outperforms imitation and reinforcement learning methods" on both benchmarks, but it does not provide specific metrics, percentages, or numerical values indicating the degree of this performance difference. 

===== k=6: 6 chunks, from pages [1, 8, 3, 8, 3, 7]
Based on the provided context, ReAct outperforms imitation and reinforcement learning methods by a significant margin in terms of success rates:

*   **On ALFWorld:** The best ReAct trial achieved an average success rate of **71%**, which is significantly higher than the best Act (imitation) trial at **45%** and the best BUTLER (reinforcement learning) trial at **37%**.
*   **On WebShop:** The text states that ReAct outperforms imitation or reinforcement learning methods trained with 103–105 task instances, but it does not 

Why does the small chain give up? Print the two chunks it was given, then the sentence in the paper (page 1) that holds the answer.

In [11]:
for doc in result_small["context"]:
    print(f"--- page {doc.metadata['page']}\n{doc.page_content}\n")

page_1 = PdfReader("documents/react_paper.pdf").pages[0].extract_text()
start = page_1.find("Furthermore, on two interactive")
end = page_1.find("examples.", start) + len("examples.")
print("--- the full sentence in the paper:\n" + page_1[start:end])

--- page 1
Furthermore, on two interactive decision making benchmarks (ALFWorld and
WebShop), ReAct outperforms imitation and reinforcement learning methods by

--- page 8
method additionally trained with 10,587 training instructions.
Results ReAct outperforms Act on both ALFWorld (Table 3) and Webshop (Table 4). On

--- the full sentence in the paper:
Furthermore, on two interactive decision making benchmarks (ALFWorld and
WebShop), ReAct outperforms imitation and reinforcement learning methods by
an absolute success rate of 34% and 10% respectively, while being prompted with
only one or two in-context examples.


**Comparing the two answers.** The paper's own sentence (page 1) says: "by an absolute success rate of 34% and 10% respectively".

- **k=2 gives up:** "it is not possible to determine by how much". Its best chunk is the right sentence, but the splitter cut it after "methods by", just before the numbers. The numbers sit in the next chunk, and that chunk was not among the two retrieved.
- **k=6 answers with numbers:** 71% for ReAct against 45% and 37% on ALFWorld (from page 8), and it works out a gap of 34 points. That matches the paper's 34% for ALFWorld. It finds no number for WebShop, so the 10% is still missing.

**A closer look at k=6.** The chunk only says "the best Act (45%) and BUTLER (37%) trials". The labels "imitation" for Act and "reinforcement learning" for BUTLER are the model's own, and the paper describes BUTLER as an imitation learning agent. More context made the answer more useful, not flawless.

So for this question k=2 was too small. The deeper cause is the chunking: chunks of 200 characters cut the key sentence in two, and a larger k only works around that.

## Exercise 3: Evaluate and compare the two configurations

Comparing answers by eye does not scale. Use the model as a judge to score each answer, then let the scores decide which `k` worked better.

Write a small `llm_judge` function that asks the model to rate something from 1 to 5, then use it to score two things for each answer:

- **Faithfulness**: is the answer supported by the retrieved context?
- **Answer relevance**: does the answer address the question?

Compare the scores for your small-`k` and large-`k` chains, and write a sentence on which you would choose and why.

Hint: build the judging prompt from the question, the context, and the answer, send it with the provided `llm`, and parse a single digit from the reply.

In [12]:
import re


def llm_judge(instruction):
    """Ask the model to rate something from 1 to 5 and return the number."""
    reply = llm.invoke(
        instruction + "\n\nReply with a single integer from 1 to 5 and nothing else."
    )
    # keep the reply only if it is exactly one digit from 1 to 5
    match = re.fullmatch(r"\s*([1-5])\s*", validate_response(reply).text)
    return int(match.group(1)) if match else None

In [13]:
def judge_result(result):
    """Score one chain result: (faithfulness, answer relevance)."""
    context = "\n---\n".join(doc.page_content for doc in result["context"])
    faithfulness = llm_judge(
        "Rate how well the answer is supported by the context. "
        "5 = every statement of the answer is in the context. "
        "1 = most of the answer is not in the context.\n\n"
        f"Context:\n{context}\n\nAnswer: {result['answer']}"
    )
    relevance = llm_judge(
        "Rate how well the answer addresses the question. "
        "5 = it answers exactly what was asked. "
        "1 = it does not answer the question.\n\n"
        f"Question: {result['input']}\n\nAnswer: {result['answer']}"
    )
    return faithfulness, relevance


scores = {"k=2": judge_result(result_small), "k=6": judge_result(result_large)}

print(f"{'chain':<8}{'faithfulness':>14}{'answer relevance':>18}")
for name, (faithfulness, relevance) in scores.items():
    print(f"{name:<8}{faithfulness!s:>14}{relevance!s:>18}")

chain     faithfulness  answer relevance
k=2                  4                 4
k=6                  4                 5


**Scores and choice.**

| chain | faithfulness | answer relevance |
|---|---:|---:|
| k=2 | 4 | 4 |
| k=6 | 4 | 5 |

The judge sees both answers as equally well supported by their context (4) and rates the k=6 answer as more relevant (5 against 4). I would choose **k=6**: it is the only one that gives numbers, and the scores point the same way.

The judge is generous to k=2, though: an answer that says "not possible to determine" still gets 4 for relevance. My own reading separates the two answers far more clearly than the judge does.

## Conclusion and Reflection

- Did citations change how much you trust the answers? When would source pages matter most?
- Which `k` gave better judge scores, and did that match your own reading of the answers?
- The judge is itself an LLM and can be biased. How would you make the evaluation more reliable, and where would a tool like RAGAS or a labelled test set help?

### My answers (our run, local model)

**Did citations change how much you trust the answers? When would source pages matter most?**

Yes. The Fever answer came with pages 5 and 6, and one look at page 6 confirmed both numbers. The closed-book answer had no source and was invented. Source pages matter most where a wrong answer is costly and the reader is able to check: medicine leaflets, legal texts, financial reports. The leaflet exercise in notebook 4 showed it: the "8 hours" answer looked fine until the page was read.

**Which `k` gave better judge scores, and did that match your own reading of the answers?**

k=6: answer relevance 5 against 4, faithfulness 4 for both. That matches my reading in direction, not in size. With k=2 the answer had no number at all. With k=6 it had the ALFWorld numbers. Neither answer found the WebShop number (10%).

**The judge is itself an LLM and can be biased. How would you make the evaluation more reliable, and where would a tool like RAGAS or a labelled test set help?**

More questions than one. A second or stronger judge model, or several judge calls per answer. A clear scale with examples in the judge prompt. And a sample of the grades checked by hand. A labelled test set (question, right answer, right passage) allows hard measures: was the right passage retrieved (hit rate, recall), and is the answer correct. RAGAS offers such metrics ready-made. Here a test set would have shown at once that with k=2 the passage with "34% and 10%" was never retrieved.